In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from datasets import *

In [2]:
# Define Autoencoder Model
class Autoencoder(nn.Module):
    def __init__(self, input_size):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_size)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [3]:
# Function to train the Autoencoder
def train_autoencoder(model, data, epochs=50, batch_size=32, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    data_loader = torch.utils.data.DataLoader(data, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        total_loss = 0
        for batch in data_loader:
            batch = batch.to(torch.float32)
            optimizer.zero_grad()
            reconstructed = model(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss:.4f}")


In [4]:
# Function to detect outage buses
def detect_outages(model, pre_outage_data, post_outage_data):
    pre_outage_data = torch.tensor(pre_outage_data, dtype=torch.float32)
    post_outage_data = torch.tensor(post_outage_data, dtype=torch.float32)

    with torch.no_grad():
        pre_recon = model(pre_outage_data)
        post_recon = model(post_outage_data)

    pre_loss = torch.mean((pre_recon - pre_outage_data) ** 2, dim=(0,1))
    post_loss = torch.mean((post_recon - post_outage_data) ** 2, dim=(0,1))

    outage_scores = post_loss - pre_loss  # Higher score means more likely outage
    outage_buses = torch.argsort(outage_scores, descending=True)+2  # Sort by score and add 2 to get bus number

    return outage_buses.numpy(), outage_scores.numpy()


In [5]:
dataset = BinaryDataset('Node123_loop')
# Example usage with random data
pre_outage_data = dataset.data_df.values[:, None, :]  # Simulating (batch, channel, feature_len)
post_outage_data = dataset.data_outage_df.values[:, None, :]  # Simulating (batch, channel, feature_len)

In [6]:
feature_len = dataset.n_buses
feature_len

122

In [7]:
# Train Model
model = Autoencoder(input_size=feature_len)
train_autoencoder(model, pre_outage_data, epochs=50)

# Detect Outage Buses
outage_buses, outage_scores = detect_outages(model, pre_outage_data, post_outage_data)

Epoch [1/50], Loss: 604.7203
Epoch [2/50], Loss: 58.6690
Epoch [3/50], Loss: 45.4144
Epoch [4/50], Loss: 42.9669
Epoch [5/50], Loss: 26.8246
Epoch [6/50], Loss: 12.6606
Epoch [7/50], Loss: 10.3068
Epoch [8/50], Loss: 10.0030
Epoch [9/50], Loss: 10.1602
Epoch [10/50], Loss: 10.1709
Epoch [11/50], Loss: 9.8523
Epoch [12/50], Loss: 9.6704
Epoch [13/50], Loss: 9.7248
Epoch [14/50], Loss: 9.7675
Epoch [15/50], Loss: 9.5711
Epoch [16/50], Loss: 10.9204
Epoch [17/50], Loss: 11.5210
Epoch [18/50], Loss: 9.8348
Epoch [19/50], Loss: 9.5532
Epoch [20/50], Loss: 9.5603
Epoch [21/50], Loss: 9.5581
Epoch [22/50], Loss: 9.6123
Epoch [23/50], Loss: 9.6233
Epoch [24/50], Loss: 9.4531
Epoch [25/50], Loss: 9.3937
Epoch [26/50], Loss: 9.4716
Epoch [27/50], Loss: 9.4702
Epoch [28/50], Loss: 9.4256
Epoch [29/50], Loss: 9.3643
Epoch [30/50], Loss: 9.5353
Epoch [31/50], Loss: 9.2251
Epoch [32/50], Loss: 8.7312
Epoch [33/50], Loss: 8.4554
Epoch [34/50], Loss: 9.8322
Epoch [35/50], Loss: 7.6558
Epoch [36/50], L

In [8]:
print("Detected Outage Buses:", outage_buses)

Detected Outage Buses: [ 74  75  76  77 120 123 121 119 122  71  70  69  68  67  65 118  64  66
  62  63  61 107  72  60 106  73  59 105  56  55  54 116 103  53 104 117
  50  51  49  52  78  48  58  79  46  47  81  80  82  57  19  45  18  20
  17  44  32  31  35 115 113  92  29 114  36  33  30  27  26  83 108  28
 102  34  10  12  85  42  16  98  97  96  37  14  13 100  93  24  15  11
 101  41  84  43  40  95   7  39  94 112  25  21  23  89  99  22   6  87
   3  88   5  90  86  38   9   2   4  91   8 110 111 109]
